In [14]:
from config import API_KEY, get_connection

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT last_run_timestamp FROM raw.pipeline_metadata WHERE pipeline_name = 'tmdb_incremental_sync'
        """)
        last_run = cur.fetchone()[0]
        print(last_run)

2026-05-15 00:00:00


In [15]:
import datetime
import requests
import psycopg2
from psycopg2.extras import Json

all_ids = set()

start_date = last_run.date()
end_date = datetime.date.today()

while start_date <= end_date:

    chunk_end = min(
        start_date + datetime.timedelta(days=6),
        end_date
    )

    current_page = 1

    while True:
        url = "https://api.themoviedb.org/3/movie/changes"

        params = {
            "api_key": API_KEY,
            "start_date": start_date.strftime("%Y-%m-%d"),
            "end_date": chunk_end.strftime("%Y-%m-%d"),
            "page": current_page
        }

        response = requests.get(url, params=params).json()

        if not response.get("success", True):
            print("API Error:", response)
            break

        all_ids.update(
            movie["id"]
            for movie in response.get("results", [])
        )

        total_pages = response.get("total_pages", 0)

        if current_page >= total_pages:
            break

        current_page += 1

    start_date = chunk_end + datetime.timedelta(days=1)

all_ids = list(all_ids)

print(f"Found {len(all_ids)} unique movie IDs")
print(all_ids)

Found 103331 unique movie IDs
[3, 5, 262149, 1048586, 11, 12, 13, 14, 15, 16, 1048587, 18, 19, 20, 1310741, 22, 1048598, 24, 25, 27, 28, 33, 35, 38, 786474, 786477, 524339, 55, 58, 59, 524348, 62, 63, 64, 65, 66, 1572929, 68, 67, 70, 71, 69, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 262227, 85, 524369, 87, 88, 89, 90, 91, 93, 262237, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 786535, 1310830, 111, 110, 262254, 114, 115, 116, 109, 118, 117, 120, 121, 122, 123, 1572989, 127, 128, 129, 1572992, 262272, 132, 524417, 134, 786563, 136, 137, 138, 139, 133, 1310861, 141, 142, 143, 145, 146, 524434, 147, 149, 150, 144, 152, 153, 154, 155, 524435, 157, 1048731, 158, 1310871, 161, 162, 163, 164, 165, 166, 524453, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 1048746, 180, 182, 184, 185, 186, 187, 1573051, 189, 192, 193, 194, 262336, 196, 197, 1048771, 199, 195, 201, 786630, 203, 204, 205, 200, 207, 1048780, 1310925, 262354, 211, 1048786, 213, 214, 215, 216, 217, 2

In [10]:
len(all_ids)

101678